# 04 MLP 모델 학습

- MLP 기반 이진 분류 모델을 학습한다.
- 학습 과정에서 SMOTE와 5-FOLD를 통해 데이터 불균형을 막고 학습을 검증한다.
- 각각의 검증 과정마다 SMOTE를 진행한다.

### 5-Fold 교차검증에서 SMOTE 적용 방식
- SMOTE는 검증 데이터나 테스트 데이터에 적용하면 데이터 누수(Data Leakage)가 발생할 수 있으므로, 반드시 학습 데이터에만 적용해야 한다.
- 5-Fold 교차검증에서는 전체 학습 데이터를 5개의 fold로 나누고, 매 반복마다 4개의 fold는 학습 데이터로, 나머지 1개의 fold는 검증 데이터로 사용한다.  
- 따라서 각 반복마다 검증 데이터가 달라지므로, SMOTE도 매 fold마다 해당 fold의 학습 데이터에만 새로 적용해야 한다.

즉, 전체 데이터에 SMOTE를 먼저 적용한 뒤 5-Fold를 수행하는 것이 아니라, 다음과 같은 순서로 진행한다.

1. 원본 데이터를 학습 데이터와 최종 테스트 데이터로 분리한다.
2. 학습 데이터에 대해 5-Fold를 수행한다.
3. 각 fold에서 학습 fold와 검증 fold를 분리한다.
4. SMOTE는 학습 fold에만 적용한다.
5. 검증 fold는 원본 상태 그대로 평가에 사용한다.
6. 모든 fold의 성능을 평균내어 모델의 검증 성능을 확인한다.

## 1. 학습용 라이브러리 로드


In [1]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. 데이터 로드 및 입력/라벨 분리

- 03번에서 생성한 final_hybrid_*.csv를 사용한다. 
- validation/test는 학습에 사용하지 않는다.
- SEED 랜덤 고정값
- 데이터를 읽고, MLP에 쓸 입력 x와 정답 y를 분리한다.

In [2]:
SEED = 42
TARGET_COL = 'target_label'

train_df = pd.read_csv('csv/final_hybrid_train.csv')
val_df = pd.read_csv('csv/final_hybrid_val.csv')
test_df = pd.read_csv('csv/final_hybrid_test.csv')

feature_cols = [col for col in train_df.columns if col != TARGET_COL]
X_train_all = train_df[feature_cols]
y_train_all = train_df[TARGET_COL].astype(int)

X_val = val_df[feature_cols]
y_val = val_df[TARGET_COL].astype(int)

X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL].astype(int)

print(f"학습용 데이터: ",train_df.shape)
print(f"검증용 데이터: ",val_df.shape)
print(f"테스트용 데이터: ",test_df.shape)

print(f"정답 데이터 분포도 \n0: 일반 리뷰 | 1: 이벤트 리뷰")
print(y_train_all.value_counts().sort_index().to_dict())

학습용 데이터:  (6188, 148)
검증용 데이터:  (1326, 148)
테스트용 데이터:  (1326, 148)
정답 데이터 분포도 
0: 일반 리뷰 | 1: 이벤트 리뷰
{0: 3983, 1: 2205}


## 3. 후보 MLP 모델 정의

- 최종 분류 모델은 Scikit-learn `MLPClassifier`를 채택한다.
- 입력 데이터는 03번에서 생성한 PCA 기반 KcBERT feature와 메타데이터가 결합된 숫자형 벡터이다.
- 목표는 각 리뷰가 일반 리뷰인지 이벤트성 리뷰인지 판별하는 이진 분류 문제이다.
- 하나의 MLP 구조만 고정하지 않고, 은닉층 크기와 깊이를 다르게 설정한 여러 후보 모델을 비교한다.
- 이후 성능 평가를 통해, 가장 좋은 모델 형태를 최종 모델로 설정한다.

후보 모델은 다음과 같다.
1. `mlp_32`: 은닉층 1개, 뉴런 32개
2. `mlp_64`: 은닉층 1개, 뉴런 64개
3. `mlp_128`: 은닉층 1개, 뉴런 128개
4. `mlp_128_64`: 은닉층 2개, 첫 번째 128개, 두 번째 64개
5. `mlp_256_128`: 은닉층 2개, 첫 번째 256개, 두 번째 128개
6. `mlp_128_64_alpha_1e_3`: 은닉층 구조는 `128, 64`로 동일하지만, `alpha` 값을 더 크게 설정하여 정규화 강도를 높인 모델

- alpha 값이 작은 경우: 모델의 학습 자율성 증가, 과적합 위험 증가
- alpha 값이 큰 경우: 모델의 학습 자율성 감소, 과적합 위험 감소, 다만 학습이 덜 될 수도 있다.

In [3]:
model_configs = [
    {'name': 'mlp_32', 'hidden_layer_sizes': (32,), 'alpha': 1e-4},
    {'name': 'mlp_64', 'hidden_layer_sizes': (64,), 'alpha': 1e-4},
    {'name': 'mlp_128', 'hidden_layer_sizes': (128,), 'alpha': 1e-4},
    {'name': 'mlp_128_64', 'hidden_layer_sizes': (128, 64), 'alpha': 1e-4},
    {'name': 'mlp_256_128', 'hidden_layer_sizes': (256, 128), 'alpha': 1e-4},
    {'name': 'mlp_128_64_alpha_1e_3', 'hidden_layer_sizes': (128, 64), 'alpha': 1e-3},
]

def make_model(config: dict, random_state: int) -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=config['hidden_layer_sizes'],
            activation='relu',
            alpha=config['alpha'],
            batch_size=64,
            learning_rate_init=1e-3,
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=random_state,
        )),
    ])


## 4. SMOTE + 5-Fold 교차검증

- 5-Fold 교차검증은 train 데이터 안에서 후보 MLP 모델들의 성능을 비교하기 위한 학습/검증 단계이다.
- 먼저 train 데이터를 5개의 fold로 나눈 뒤, 각 반복에서 4개 fold는 학습용, 1개 fold는 검증용으로 사용한다.
- 클래스 불균형 보정을 위해 각 반복의 학습 fold에만 SMOTE를 적용한다.
- 검증 fold에는 SMOTE를 적용하지 않고, 원본 분포 그대로 성능을 평가한다.
- 앞서 정의한 여러 MLP 후보 모델에 동일한 교차검증 과정을 적용한다.
- 검증 데이터에서 이벤트 리뷰일 확률이 0.5 이상이면 이벤트성 리뷰로 판단한다.
- 이후 각 모델과 fold별 성능 평가 지표를 저장하고, 모델별 평균 성능을 계산한다.

저장하는 지표는 다음과 같다.

1. `f1`: precision과 recall의 균형을 나타내는 지표
2. `pr_auc`: 이벤트 리뷰처럼 관심 클래스의 탐지 성능을 볼 때 유용한 지표
3. `precision`: 이벤트 리뷰라고 예측한 것 중 실제 이벤트 리뷰의 비율
4. `recall`: 실제 이벤트 리뷰 중 모델이 이벤트 리뷰로 잡아낸 비율
5. `roc_auc`: 전체적인 이진 분류 구분 능력을 나타내는 지표

- 이벤트라고 찍은 예측의 신뢰도를 위해 F1을 우선으로 보고 PR-AUC를 보조 기준으로 둔다.

In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []

for config in model_configs:
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_all, y_train_all), start=1):
        X_fold_train = X_train_all.iloc[train_idx]
        y_fold_train = y_train_all.iloc[train_idx]
        X_fold_valid = X_train_all.iloc[valid_idx]
        y_fold_valid = y_train_all.iloc[valid_idx]

        X_smote, y_smote = SMOTE(random_state=SEED).fit_resample(X_fold_train, y_fold_train)
        model = make_model(config, random_state=SEED + fold)
        model.fit(X_smote, y_smote)

        valid_prob = model.predict_proba(X_fold_valid)[:, 1]
        valid_pred = (valid_prob >= 0.5).astype(int)

        cv_rows.append({
            'model': config['name'],
            'fold': fold,
            'hidden_layer_sizes': str(config['hidden_layer_sizes']),
            'alpha': config['alpha'],
            'train_before_label_0': int((y_fold_train == 0).sum()),
            'train_before_label_1': int((y_fold_train == 1).sum()),
            'train_after_smote_label_0': int((pd.Series(y_smote) == 0).sum()),
            'train_after_smote_label_1': int((pd.Series(y_smote) == 1).sum()),
            'valid_label_0': int((y_fold_valid == 0).sum()),
            'valid_label_1': int((y_fold_valid == 1).sum()),
            'f1': float(f1_score(y_fold_valid, valid_pred)),
            'pr_auc': float(average_precision_score(y_fold_valid, valid_prob)),
            'precision': float(precision_score(y_fold_valid, valid_pred, zero_division=0)),
            'recall': float(recall_score(y_fold_valid, valid_pred, zero_division=0)),
            'roc_auc': float(roc_auc_score(y_fold_valid, valid_prob)),
        })

cv_results = pd.DataFrame(cv_rows)
cv_summary = (
    cv_results.groupby('model')
    .agg(
        mean_f1=('f1', 'mean'),
        std_f1=('f1', 'std'),
        mean_pr_auc=('pr_auc', 'mean'),
        std_pr_auc=('pr_auc', 'std'),
        mean_precision=('precision', 'mean'),
        mean_recall=('recall', 'mean'),
        mean_roc_auc=('roc_auc', 'mean'),
    )
    .reset_index()
    .sort_values(['mean_f1', 'mean_pr_auc'], ascending=False)
)

cv_results.to_csv('outputs/proposed_mlp_cv_results.csv', index=False, encoding='utf-8-sig')
cv_summary.to_csv('outputs/proposed_mlp_cv_summary.csv', index=False, encoding='utf-8-sig')
cv_summary


,model,mean_f1,std_f1,mean_pr_auc,std_pr_auc,mean_precision,mean_recall,mean_roc_auc
4,mlp_32,0.433701,0.015750,0.419397,0.019028,0.413917,0.455782,0.572332
5,mlp_64,0.421443,0.011146,0.429592,0.006724,0.419512,0.423583,0.576703
0,mlp_128,0.414188,0.016217,0.428495,0.009503,0.422157,0.407256,0.578520
2,mlp_128_64_alpha_1e_3,0.406983,0.011716,0.425003,0.014151,0.422745,0.392744,0.572178
3,mlp_256_128,0.402785,0.021922,0.438858,0.004059,0.439891,0.372336,0.584559
1,mlp_128_64,0.391534,0.020261,0.426328,0.009031,0.414758,0.370975,0.570782


## 5. 최종 모델 학습 및 Validation/Test 평가

- 5-Fold 교차검증 결과에서 평균 F1-score와 PR-AUC 기준으로 가장 성능이 좋은 MLP 후보 모델을 선택한다.
- 선택된 모델 구조를 사용해 전체 train 데이터로 최종 모델을 학습한다.
- 최종 학습 단계에서도 클래스 불균형 보정을 위해 train 데이터에만 SMOTE를 적용한다.
- validation/test 데이터에는 SMOTE를 적용하지 않고, 원본 분포 그대로 평가한다.
- threshold는 일반 리뷰를 과도하게 이벤트 리뷰로 오탐하지 않도록 0.5 이상 후보에서만 탐색한다.
- 최종 모델과 평가 결과를 저장한다.

저장되는 결과는 다음과 같다.

1. `proposed_mlp_metrics.csv`: validation/test 성능 평가 결과
2. `proposed_mlp_metrics.json`: 선택된 모델 설정과 성능 평가 결과
3. `proposed_mlp_selected_config.json`: 최종 선택된 MLP 모델 설정
4. `proposed_mlp_final_model.joblib`: 최종 학습된 모델과 재사용에 필요한 설정


### 5-1. 최종 모델 선택 및 학습

- 교차검증 요약표에서 가장 위에 있는 후보 모델을 선택하고, 전체 train 데이터에 SMOTE를 적용한 뒤 최종 모델을 학습한다.


In [5]:
best_model_name = cv_summary.iloc[0]['model']
best_config = next(config for config in model_configs if config['name'] == best_model_name)

X_train_smote, y_train_smote = SMOTE(random_state=SEED).fit_resample(X_train_all, y_train_all)

final_model = make_model(best_config, random_state=SEED)
final_model.fit(X_train_smote, y_train_smote)

val_prob = final_model.predict_proba(X_val)[:, 1]
test_prob = final_model.predict_proba(X_test)[:, 1]

### 5-2. Threshold 탐색

- 기본 threshold는 0.5이다. 
- 추가로 validation 데이터에서 F1-score가 가장 높은 threshold를 찾되, 너무 낮은 기준이 선택되지 않도록 후보를 0.5 이상으로 제한한다.


In [6]:
MIN_TUNED_THRESHOLD = 0.5

precisions, recalls, thresholds = precision_recall_curve(y_val, val_prob)

if len(thresholds) == 0:
    best_threshold = MIN_TUNED_THRESHOLD
    threshold_candidates = pd.DataFrame()
else:
    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    threshold_candidates = pd.DataFrame({
        'threshold': thresholds,
        'precision': precisions[:-1],
        'recall': recalls[:-1],
        'f1': f1s,
    })
    threshold_candidates = threshold_candidates[threshold_candidates['threshold'] >= MIN_TUNED_THRESHOLD]

    if threshold_candidates.empty:
        best_threshold = MIN_TUNED_THRESHOLD
    else:
        best_threshold = float(
            threshold_candidates.sort_values('f1', ascending=False).iloc[0]['threshold']
        )

threshold_rank = threshold_candidates.sort_values('f1', ascending=False).head(10).reset_index(drop=True)
threshold_rank.insert(0, 'rank', threshold_rank.index + 1)
threshold_rank


,rank,threshold,precision,recall,f1
0,1,0.507434,0.395644,0.461864,0.426197
1,2,0.507277,0.394928,0.461864,0.425781
2,3,0.500889,0.393178,0.463983,0.425656
3,4,0.506727,0.394213,0.461864,0.425366
4,5,0.508421,0.395264,0.459746,0.425073
5,6,0.501435,0.393502,0.461864,0.424951
6,7,0.507975,0.394545,0.459746,0.424658
7,8,0.500989,0.392086,0.461864,0.424125
8,9,0.510546,0.394161,0.457627,0.423529
9,10,0.524474,0.398496,0.449153,0.422311


### 5-3. 성능 계산 및 결과 저장

- 기본 threshold 0.5와 validation 기준 조정 threshold 0.507434를 각각 적용하여 validation/test 성능을 비교한다. 
- 이후 성능표, 선택된 설정, 최종 학습 모델을 저장한다.


In [7]:
def metric_row(split: str, y_true, prob, threshold: float) -> dict:
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'split': split,
        'threshold': float(threshold),
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


metrics = pd.DataFrame([
    metric_row('validation_default_0_5', y_val, val_prob, 0.5),
    metric_row('validation_tuned_min_0_5', y_val, val_prob, best_threshold),
    metric_row('test_default_0_5', y_test, test_prob, 0.5),
    metric_row('test_tuned_min_0_5', y_test, test_prob, best_threshold),
])

selected = {
    'selected_model': best_config['name'],
    'hidden_layer_sizes': list(best_config['hidden_layer_sizes']),
    'alpha': best_config['alpha'],
    'selection_rule': 'highest mean_f1, then mean_pr_auc on 5-fold CV',
    'min_threshold_for_tuning': MIN_TUNED_THRESHOLD,
    'best_threshold_from_validation': best_threshold,
}

metrics.to_csv('outputs/proposed_mlp_metrics.csv', index=False, encoding='utf-8-sig')
with open('outputs/proposed_mlp_metrics.json', 'w', encoding='utf-8') as f:
    json.dump({'selected_config': selected, 'metrics': metrics.to_dict(orient='records')}, f, ensure_ascii=False, indent=2)
with open('outputs/proposed_mlp_selected_config.json', 'w', encoding='utf-8') as f:
    json.dump(selected, f, ensure_ascii=False, indent=2)

model_bundle = {
    'model': final_model,
    'feature_cols': feature_cols,
    'target_col': TARGET_COL,
    'selected_config': selected,
    'default_threshold': 0.5,
    'best_threshold': best_threshold,
}
joblib.dump(model_bundle, 'outputs/proposed_mlp_final_model.joblib')


['outputs/proposed_mlp_final_model.joblib']

### 5-4. 결과 표 출력

- 저장된 성능 결과를 노트북에서 바로 읽을 수 있도록 표 형태로 출력한다.


In [8]:
selected_display = pd.DataFrame([{
    '선택 모델': selected['selected_model'],
    '은닉층 구조': str(tuple(selected['hidden_layer_sizes'])),
    'alpha': selected['alpha'],
    '선택 기준': selected['selection_rule'],
    'threshold 최소 기준': selected['min_threshold_for_tuning'],
    'validation 최적 threshold': selected['best_threshold_from_validation'],
}])

threshold_display = pd.DataFrame([
    {'기준': 'default', 'threshold': 0.5, '설명': '확률이 0.5 이상이면 이벤트 리뷰로 예측'},
    {'기준': 'tuned_min_0_5', 'threshold': best_threshold, '설명': '0.5 이상 후보 중 validation F1-score가 가장 높았던 기준'},
])

metrics_display = metrics.rename(columns={
    'split': '평가 구분',
    'threshold': 'threshold',
    'f1': 'F1',
    'pr_auc': 'PR-AUC',
    'precision': 'Precision',
    'recall': 'Recall',
    'roc_auc': 'ROC-AUC',
    'tn': 'TN(일반→일반)',
    'fp': 'FP(일반→이벤트)',
    'fn': 'FN(이벤트→일반)',
    'tp': 'TP(이벤트→이벤트)',
})
metrics_display = metrics_display.round({
    'threshold': 4,
    'F1': 4,
    'PR-AUC': 4,
    'Precision': 4,
    'Recall': 4,
    'ROC-AUC': 4,
})

test_pred_tuned = (test_prob >= best_threshold).astype(int)
report_display = (
    pd.DataFrame(classification_report(
        y_test,
        test_pred_tuned,
        digits=4,
        zero_division=0,
        output_dict=True,
    ))
    .T
    .reset_index()
    .rename(columns={'index': '구분', 'precision': 'Precision', 'recall': 'Recall', 'f1-score': 'F1', 'support': '개수'})
)
report_display['구분'] = report_display['구분'].replace({
    '0': '일반 리뷰(0)',
    '1': '이벤트 리뷰(1)',
    'accuracy': '정확도',
    'macro avg': '단순 평균',
    'weighted avg': '가중 평균',
})
report_display = report_display.round({'Precision': 4, 'Recall': 4, 'F1': 4, '개수': 0})

print('최종 선택 모델')
display(selected_display)
print('threshold 기준')
display(threshold_display)
print('Validation/Test 성능 비교')
display(metrics_display)
print('Test 데이터 상세 리포트(임계값이 0.5이상 후보 중 F1이 가장 높은 threshold 기준)')
display(report_display)


최종 선택 모델


,선택 모델,은닉층 구조,alpha,선택 기준,threshold 최소 기준,validation 최적 threshold
0,mlp_32,"(32,)",0.0001,"highest mean_f1, then mean_pr_auc on 5-fold CV",0.5,0.507434


threshold 기준


,기준,threshold,설명
0,default,0.500000,확률이 0.5 이상이면 이벤트 리뷰로 예측
1,tuned_min_0_5,0.507434,0.5 이상 후보 중 validation F1-score가 가장 높았던 기준


Validation/Test 성능 비교


,평가 구분,threshold,F1,PR-AUC,Precision,Recall,ROC-AUC,TN(일반→일반),FP(일반→이벤트),FN(이벤트→일반),TP(이벤트→이벤트)
0,validation_default_0_5,0.5000,0.4257,0.3952,0.3932,0.4640,0.5539,516,338,253,219
1,validation_tuned_min_0_5,0.5074,0.4262,0.3952,0.3956,0.4619,0.5539,521,333,254,218
2,test_default_0_5,0.5000,0.4290,0.3808,0.3864,0.4820,0.5374,491,362,245,228
3,test_tuned_min_0_5,0.5074,0.4274,0.3808,0.3879,0.4757,0.5374,498,355,248,225


Test 데이터 상세 리포트(임계값이 0.5이상 후보 중 F1이 가장 높은 threshold 기준)


,구분,Precision,Recall,F1,개수
0,일반 리뷰(0),0.6676,0.5838,0.6229,853.0
1,이벤트 리뷰(1),0.3879,0.4757,0.4274,473.0
2,정확도,0.5452,0.5452,0.5452,1.0
3,단순 평균,0.5277,0.5298,0.5251,1326.0
4,가중 평균,0.5678,0.5452,0.5531,1326.0


## 6. 결과 해석 및 다음 단계 연결

04번의 목적은 최종 프로젝트 모델을 확정하는 것이 아니라, 현재 제안 방식인 `KcBERT PCA feature + metadata + MLP` 구조를 학습하고 후보 모델 중 가장 나은 모델을 저장하는 것이다.

### 6-1. 현재 결과 해석

- 5-Fold 교차검증 기준으로 가장 높은 평균 F1을 보인 모델은 `mlp_32`이다.
- 선택 기준은 `mean_f1`을 우선으로 보고, 동률 또는 유사한 경우 `mean_pr_auc`를 보조 기준으로 보는 방식이다.
- validation 데이터에서 threshold 후보를 0.5 이상으로 제한한 뒤 F1이 가장 높은 값을 찾았고, 선택된 threshold는 약 `0.5074`이다.
- test 데이터 기준 이벤트 리뷰(1)의 F1-score는 `0.4274`로 나타났다.
- 일반 리뷰(0)의 F1-score는 `0.6229`로, 이벤트 리뷰보다 일반 리뷰를 더 안정적으로 구분한다.
- 즉, 현재 Proposed MLP는 이벤트 리뷰를 어느 정도 잡아내지만, 이벤트 리뷰 탐지 성능이 충분히 높다고 보기는 어렵다.

### 6-2. 현재 모델의 의미

- `outputs/proposed_mlp_final_model.joblib`에는 현재 학습된 Proposed MLP 모델과 입력 feature 컬럼, target 컬럼, 선택된 설정, threshold 정보가 함께 저장된다.
- 이 모델은 별점 정제 단계에서 바로 불러와 사용할 수 있는 후보 모델이다.
- 다만 아직 베이스라인과 비교하지 않았으므로, 프로젝트의 최종 모델로 확정된 것은 아니다.
- 따라서 04번의 결과는 `최종 모델 확정`이 아니라 `Proposed MLP 후보 모델 학습 완료`로 해석해야 한다.

### 6-3. 05번과의 연결: 베이스라인 비교 및 메타데이터 효과 검증

05번에서는 현재 Proposed MLP가 실제로 좋은 모델인지 확인하기 위해 베이스라인 모델과 비교해야 한다. 이 단계의 핵심은 단순히 점수만 비교하는 것이 아니라, `KcBERT 임베딩 결과에 메타데이터를 결합한 전략이 탐지율을 얼마나 향상시켰는지`를 검증하는 것이다.

비교해야 할 모델은 다음과 같다.

1. `TF-IDF + Random Forest`
2. `KcBERT PCA only + MLP`
3. `Metadata only + MLP`
4. `KcBERT PCA + Metadata Hybrid + MLP`

비교 지표는 04번과 동일하게 `F1`, `PR-AUC`, `Precision`, `Recall`, `ROC-AUC`를 사용한다. 이렇게 해야 04번의 Proposed MLP 결과와 05번의 baseline 결과를 같은 기준으로 비교할 수 있다.

05번에서 확인할 내용은 다음과 같다.

- Hybrid MLP가 TF-IDF + Random Forest보다 좋은지 확인한다.
- Hybrid MLP가 KcBERT PCA only 모델보다 좋은지 확인한다.
- metadata only 모델이 어느 정도 성능을 내는지 확인한다.
- 메타데이터를 결합했을 때 F1, PR-AUC, Recall이 실제로 개선되는지 확인한다.
- 모델 예측에 큰 영향을 준 메타데이터가 무엇인지 분석한다.

### 6-4. 06번과의 연결: 오답 분석 및 최종 모델 선택

- 06번에서는 04번의 Proposed MLP 결과와 05번의 baseline 결과를 비교하고, 오답 리뷰를 따로 추출해 원인을 분석한다.
- 일반 리뷰를 이벤트 리뷰로 잘못 예측한 FP 사례와 이벤트 리뷰를 일반 리뷰로 놓친 FN 사례를 나눠서 본다.
- KcBERT가 은어, 신조어, 우회 표현을 처리할 때 어떤 한계를 보이는지 확인한다.
- 메타데이터가 텍스트 기반 모호성을 줄이는 데 실제로 도움을 줬는지 확인한다.
- 최종 모델 선택 기준은 이벤트 리뷰(1)의 F1과 PR-AUC를 중심으로 두고, precision/recall 균형과 오답 분석 결과도 함께 본다.

### 6-5. 07번과의 연결: 최종 모델 기반 별점 정제

- 07번에서는 06번에서 선택된 최종 모델을 불러와 전체 리뷰 데이터에 이벤트 리뷰 확률을 예측한다.
- 그 예측 결과를 이용해 이벤트성 리뷰로 판단된 행을 찾고, 별점 정제 전/후를 비교한다.
- 따라서 04번에서는 전체 리뷰 예측을 미리 수행하지 않고, 모델과 평가 결과만 저장하는 것이 맞다.

정리하면, 04번은 `Proposed MLP 학습 및 저장`까지 완료된 상태이며, 다음 단계는 베이스라인 비교, 메타데이터 효과 검증, 오답 분석을 끝낸 뒤 최종 모델을 선택하는 것이다.
